In [1]:
import uproot
import ROOT
import numpy as np
import matplotlib.pyplot as plt
import awkward as ak

In [2]:
POT_MC = 2.43007e+20
POT_OFFBEAM = 2.11e+20

In [ ]:
mode_vars = ['STANDARD']
mode_vars = ['STANDARD','GAIN','BETA','ALPHA','R']
var_color = ['gray','blue','deepskyblue','darkgreen','lime','red','orange','purple','pink']

In [ ]:
for RR in np.arange(2,25,1) :

    fig = plt.figure()

    gs = fig.add_gridspec(2, 1, height_ratios=[8,2], hspace=0)

    print('RR interval: [',RR-1,RR,')')

    filename = 'varMC_DATA.root'
    file = uproot.open(filename)
    tree = file['tree']
    print("num entries:", tree.num_entries)
    print("keys:", tree.keys())
    arrays = tree.arrays(
            ["slice/_protons/_protons._dedx", 
            "slice/_protons/_protons._rr",
            "slice/_mu/_mu._dedx", 
            "slice/_mu/_mu._rr"],
            library="ak"
            )
    
    #dedx_thisRR = []

    #for i in range(len(arrays)):
    #    slice = arrays[i]

    #    protons_dedx = slice['_protons._dedx'] #array di vettori di dedx, uno per protone nella slice
    #   protons_rr = slice['_protons._rr']

    #    for rr_vec,dedx_vec in zip(protons_rr,protons_dedx): #vettore rr e dedx del protone in esame nel ciclo
    #        for rr,dedx in zip(rr_vec,dedx_vec):
    #            if rr >= RR - 1 and rr < RR :
    #                dedx_thisRR.append(dedx)

    rr_flat   = ak.to_numpy(ak.flatten(arrays["slice/_protons/_protons._rr"],   axis=None))
    print(rr_flat)
    dedx_flat = ak.to_numpy(ak.flatten(arrays["slice/_protons/_protons._dedx"], axis=None))
    mask = (rr_flat >= RR - 1) & (rr_flat < RR)
    dedx_thisRR = dedx_flat[mask]
    
    h_data = ROOT.TH1D(f"h_data_rr{RR}","",100,0,30)
    h_data.Sumw2()
    data_np = np.array(dedx_thisRR)
    #h_data.FillN(data_np.size,data_np,np.ones_like(data_np))
    for hit_data in data_np : h_data.Fill(hit_data)

    h_data.Scale(1. / h_data.Integral("width"))

    bin_centers_DATA = []
    counts_DATA = []
    errors_DATA = []

    for bin in range(1,100 + 1):
        bin_centers_DATA.append(h_data.GetBinCenter(bin))
        counts_DATA.append(h_data.GetBinContent(bin))
        errors_DATA.append(h_data.GetBinError(bin))


    filename = 'varMC_OFFBEAM.root'
    file = uproot.open(filename)
    tree = file['tree']
    arrays = tree.arrays(
            ["slice/_protons/_protons._dedx", 
            "slice/_protons/_protons._rr",
            "slice/_mu/_mu._dedx", 
            "slice/_mu/_mu._rr"],
            library="ak"
            )
    
    rr_flat   = ak.to_numpy(ak.flatten(arrays["slice/_protons/_protons._rr"],   axis=None))
    dedx_flat = ak.to_numpy(ak.flatten(arrays["slice/_protons/_protons._dedx"], axis=None))
    mask = (rr_flat >= RR - 1) & (rr_flat < RR)
    dedx_thisRR_offbeam = dedx_flat[mask]
    weights_OFFBEAM_dedx = np.full(len(dedx_thisRR_offbeam), 1. / POT_OFFBEAM)

    plt.subplot(gs[0])
    plt.errorbar(bin_centers_DATA,counts_DATA,yerr=errors_DATA)

    for m,mode in enumerate(mode_vars):

        for n,sigma in enumerate(['_plus1sigma','_minus1sigma']):

            if n == 0 and mode == 'STANDARD' : continue
            if n == 1 and mode == 'STANDARD' : sigma = ''

            filename = f'varMC_{mode}{sigma}.root'

            file = uproot.open(filename)

            tree = file['tree']
            print(tree.keys())

            arrays = tree.arrays(
            ["slice/_protons/_protons._dedx", 
            "slice/_protons/_protons._rr",
            "slice/_mu/_mu._dedx", 
            "slice/_mu/_mu._rr"],
            library="ak"
            )

            rr_flat   = ak.to_numpy(ak.flatten(arrays["slice/_protons/_protons._rr"],   axis=None))
            dedx_flat = ak.to_numpy(ak.flatten(arrays["slice/_protons/_protons._dedx"], axis=None))
            mask = (rr_flat >= RR - 1) & (rr_flat < RR)
            dedx_thisRR = dedx_flat[mask]
            weights_MC_dedx = np.full(len(dedx_thisRR), 1. / POT_MC)

            mc_off = np.concatenate([dedx_thisRR, dedx_thisRR_offbeam])
            weights_mc_off = np.concatenate([weights_MC_dedx, weights_OFFBEAM_dedx])
            n_mc_off, bins = np.histogram(
            mc_off,
            bins=100,
            range=(0, 30),
            weights=weights_mc_off,
            density=True
            )

            bin_centers = 0.5 * (bins[:-1] + bins[1:])
            bin_widths = np.diff(bins)

            ratio = np.zeros_like(counts_DATA)
            ratio_err = np.zeros_like(counts_DATA)

            for k in range(len(counts_DATA)):
                if n_mc_off[k] > 0:
                    ratio[k] = counts_DATA[k] / n_mc_off[k]

            sigma_n = 0
            if sigma == '_plus1sigma' : sigma_n = 1
            elif sigma == '_minus1sigma' : sigma_n = -1

            plt.subplot(gs[0])
            plt.hist(mc_off, bins=100, range=(0, 30), weights=weights_mc_off, density=True, histtype='step', lw=2, label=fr'MC var {mode} ${sigma_n}\sigma$', color = var_color[m])
            plt.setp(plt.gca().get_xticklabels(), visible=False)
            plt.gca().tick_params(labelbottom=False)
            plt.ylabel('entries (area norm.)', fontsize=14)
            plt.yticks(fontsize=14)

            plt.title('dE/dx for RR bin [{},{}) cm'.format(RR,RR+1), fontsize=18)

            plt.subplot(gs[1])

            dx = bin_centers[1] - bin_centers[0]

            edges = np.concatenate([
            [bin_centers[0] - dx/2],
            bin_centers[:-1] + dx/2,
            [bin_centers[-1] + dx/2]
            ])

            plt.stairs(ratio,edges,fill=False,lw=2,color=var_color[m])

            plt.fill_between(
            edges[:-1],
            1,
            ratio,
            step='post',
            alpha=0.4,
            color=var_color[m],
            lw=0
            )

            plt.axhline(1.0, color='black', linestyle='--')

            plt.xlabel('dE/dx [MeV/cm]', fontsize=14)
            plt.ylabel('DATA/MC', fontsize=14)
            plt.xticks(fontsize=14, rotation=60)
            plt.yticks(fontsize=14)


    plt.legend()
    plt.savefig(f'rr{RR}.pdf',format='pdf',bbox_inches='tight')


        


RR interval: [ 1 2 )
num entries: 125011
keys: ['slice', 'slice/TObject', 'slice/TObject/fUniqueID', 'slice/TObject/fBits', 'slice/_run', 'slice/_evt', 'slice/_slice_counter', 'slice/_protons', 'slice/_protons/_protons.fUniqueID', 'slice/_protons/_protons.fBits', 'slice/_protons/_protons._lr', 'slice/_protons/_protons._proba', 'slice/_protons/_protons._depE', 'slice/_protons/_protons._KE', 'slice/_protons/_protons._length', 'slice/_protons/_protons._daughter_vars', 'slice/_protons/_protons._dedx', 'slice/_protons/_protons._rr', 'slice/_protons/_protons._theta_xw', 'slice/_mu', 'slice/_mu/_mu.TObject', 'slice/_mu/_mu.TObject/_mu.fUniqueID', 'slice/_mu/_mu.TObject/_mu.fBits', 'slice/_mu/_mu._lr', 'slice/_mu/_mu._proba', 'slice/_mu/_mu._depE', 'slice/_mu/_mu._KE', 'slice/_mu/_mu._length', 'slice/_mu/_mu._daughter_vars', 'slice/_mu/_mu._dedx', 'slice/_mu/_mu._rr', 'slice/_mu/_mu._theta_xw', 'slice/_reco_class', 'slice/_true_class', 'slice/_nuE', 'slice/_mu_pro_angle', 'slice/_nu_score', 's

ZeroDivisionError: float division by zero

<Figure size 640x480 with 0 Axes>